# Native SAM3 verbose-prompt counting (Kaggle)

Runs the standalone `sam3-verbose-counting/` pipeline end to end: environment setup, gated `sam3.pt` checkpoint download, and text-guided object counting with inline visualizations.

SAM3 takes a **verbose natural-language prompt** (e.g. *"all pairs of black sunglasses displayed on the retail rack"*), returns text-grounded bounding boxes + confidence scores, and the **count is the number of detections above a threshold**.

The notebook also demonstrates the **overlap filter**: SAM3 cannot tell whether a sunglasses pair is *worn* vs *displayed* from text alone, so we run a second pass (e.g. `faces of people`) and drop every target box whose center falls inside a face/person detection.

**Requirements on Kaggle:**
- GPU accelerator enabled (P100/T4 or better; T4 works with fp16 AMP).
- A Kaggle notebook **Secret** named `HF_TOKEN` with an approved token for the gated [`facebook/sam3`](https://huggingface.co/facebook/sam3) model (request access there first). No `hf auth login` needed.

## 1. Setup: clone the repo and install SAM3 runtime deps

Edit `REPO_URL` to point at your actual repository. The cell **clones the repo on first run and pulls the latest changes on every subsequent run**, so the notebook always picks up new commits. It then checks the few SAM3 runtime dependencies that are not guaranteed to ship with the Kaggle base kernel (`timm`, `einops`, `ftfy`, ...) and installs only what is missing into the notebook kernel, and makes `sam3-verbose-counting/` importable.

In [ ]:
import importlib
import importlib.util
import shutil
import subprocess
import sys
from pathlib import Path

# --- 1a. Clone the repository (edit this URL) ------------------------------
REPO_URL = "https://github.com/fez-Ox/pxModel-Object-Counting.git"  # <-- update me
REPO_DIR = Path("/kaggle/working/pxModel-localization")
SAM3_APP = REPO_DIR / "sam3-verbose-counting"

if not (REPO_DIR / ".git").is_dir():
    if REPO_DIR.exists():
        print(f"Removing stale non-git directory: {REPO_DIR}")
        shutil.rmtree(REPO_DIR)
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Updating existing repository at {REPO_DIR}...")
    branch = subprocess.run(
        ["git", "-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all"], check=True)
    subprocess.run(
        ["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{branch}"],
        check=True,
    )
    print(f"Repository synced to origin/{branch}.")

# --- 1b. Install only the SAM3 deps missing from the Kaggle kernel ---------
missing = []
for name in ["torch", "torchvision", "PIL", "numpy", "timm", "einops",
             "ftfy", "regex", "wrapt", "typing_extensions"]:
    try:
        importlib.import_module(name)
    except ImportError:
        missing.append(name)
if "PIL" in missing:
    missing[missing.index("PIL")] = "Pillow"
if missing:
    print("Installing missing deps:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=True)

# --- 1c. Make the standalone app importable --------------------------------
sys.path.insert(0, str(SAM3_APP))
print("SAM3 app ready:", SAM3_APP)
print("Python:", sys.version.split()[0])

# --- 1d. Drop stale standalone modules + bytecode so refreshed code is used --
# If this cell is re-run in an already-warm kernel, an older `infer.py`/`sam3`
# may still be resident in sys.modules or __pycache__; purge both so the
# just-synced files are the ones imported below.
for _mod in list(sys.modules):
    if _mod == "infer" or _mod == "download_model" or _mod.startswith("sam3"):
        del sys.modules[_mod]
for _cache in SAM3_APP.rglob("__pycache__"):
    shutil.rmtree(_cache, ignore_errors=True)
importlib.invalidate_caches()
print("Standalone module caches cleared.")

# Load the native SAM3 modules by absolute path. Do not use a bare `import infer`:
# count-anything/infer.py is a different CLI module and has no filter_prompt API.
def _load_local_module(module_name, module_path):
    spec = importlib.util.spec_from_file_location(module_name, str(module_path))
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load module from {module_path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

sam3_download = _load_local_module(
    "sam3_verbose_download", SAM3_APP / "download_model.py"
)
sam3_infer = _load_local_module(
    "sam3_verbose_infer", SAM3_APP / "infer.py"
)
import inspect
print("Loaded native SAM3 infer module:", sam3_infer.__file__)
print("SAM3 infer signature:", inspect.signature(sam3_infer.Sam3VerboseCounter.infer))

## 2. Download the gated SAM3 checkpoint

The downloader first tries `HF_TOKEN` / `HUGGINGFACE_HUB_TOKEN` env vars, then the Kaggle `HF_TOKEN` notebook Secret, then any `--token` you pass. If the token is not approved for `facebook/sam3`, the checkpoint fetch returns HTTP 401 and you will see a clear message.

In [ ]:
download_model = sam3_download.download_model
DEFAULT_URL = sam3_download.DEFAULT_URL
DEFAULT_OUTPUT = sam3_download.DEFAULT_OUTPUT

sam3_path = download_model(
    url=DEFAULT_URL,
    output=DEFAULT_OUTPUT,
    force=False,
    timeout=180,
    token=None,  # auto-detects HF_TOKEN / Kaggle secret / --token
)
print("Checkpoint ready:", sam3_path)

## 3. Grab a public sample image

Downloads a COCO image (the classic "cats on a bed" sample used by `main.py`). Replace this with your own file path, folder, or image URL.

In [ ]:
import urllib.request

samples_dir = REPO_DIR / "samples"
samples_dir.mkdir(exist_ok=True)
sample = samples_dir / "coco_cats.jpg"
if not sample.exists():
    urllib.request.urlretrieve(
        "http://images.cocodataset.org/val2017/000000039769.jpg", sample
    )
print("Sample image:", sample)

## 4. Build the persistent SAM3 counter

Loads `sam3.pt` once and reuses the resident model for every image. The model is ~2.5B params, so allow a minute for weights load and the first image's forward pass.

In [ ]:
import torch

build_counter = sam3_infer.build_counter

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

counter = build_counter(threshold=0.5)  # device auto: cuda when available, else cpu

## 5. Count with a verbose prompt and show the result inline

The counter removes near-duplicate boxes and broad boxes that contain multiple separate detections. The original model boxes and cleanup counts remain in the result for auditing.

In [ ]:
from IPython.display import display

annotate = sam3_infer.annotate

prompt = "the cats resting on the bed"

result = counter.infer(sample, prompt)
annotated = annotate(sample, prompt, result["boxes"], result["scores"])

print(f"Prompt: {prompt}")
print(f"Count:  {result['count']}")
if "raw_count" in result:
    print(f"Model boxes: {result['raw_count']}  |  after cleanup: {result['deduplicated_count']}  |  "
          f"enclosing/duplicate removed: {result['redundant_box_count']}")
print(f"Inference time: {result['inference_time_seconds']:.2f}s")
if result["peak_vram_mb"] is not None:
    print(f"Peak VRAM allocated: {result['peak_vram_mb']:.1f} MiB")
    print(f"Peak VRAM reserved:  {result['peak_reserved_vram_mb']:.1f} MiB")
else:
    print("Peak VRAM: unavailable (CPU inference)")

display(annotated)

## 6. Overlap filter: exclude objects that are worn / on a person

SAM3 cannot understand "not being worn" from text. Instead, run a **second SAM3 pass** on the same image to detect the people/faces, then drop every target box whose **center falls inside a face/person box** (or whose IoU reaches a threshold).

`counter.infer(..., filter_prompt="faces of people")` returns the filtered `boxes`/`scores`/`count` and keeps the unfiltered detections under `raw_count` / `raw_boxes` / `raw_scores` for auditing.

In [ ]:
filter_prompt = "faces of people"

result_filtered = counter.infer(
    sample, prompt,
    filter_prompt=filter_prompt,
    filter_center=True,   # drop target boxes whose center is inside a face box
    filter_iou=0.0,       # set e.g. 0.3 to also drop on IoU overlap
)

print(f"Prompt: {prompt}")
print(f"Filter: {filter_prompt!r}")
print(f"Count (filtered): {result_filtered['count']}")
print(f"  model detections:      {result_filtered['raw_count']}")
print(f"  after box cleanup:      {result_filtered['deduplicated_count']}")
print(f"  enclosing/duplicate removed: {result_filtered['redundant_box_count']}")
print(f"  removed by face filter: {result_filtered['filtered_count']}")
print(f"  filter objects found:  {result_filtered['filter_object_count']}")

annotated_filtered = annotate(
    sample, prompt, result_filtered["boxes"], result_filtered["scores"]
)
display(annotated_filtered)

> On the sample cats image there are no people, so nothing is removed (`removed = 0`). Test this on your own retail/street photos below — any sunglasses overlapping a face detection will be excluded from the count.

## 7. Localize each brand separately

Pass one text prompt per brand. SAM3 encodes the image once, then reuses the image features for every brand prompt. The result and visualization keep each brand's boxes and count separate.

In [ ]:
brand_prompts = {
    "Ray-Ban": "Ray-Ban sunglasses displayed on the retail rack",
    "Oakley": "Oakley sunglasses displayed on the retail rack",
}
brand_filter_prompt = None  # optionally set to "faces of people"

brand_result = counter.infer_brands(
    sample,
    brand_prompts,
    filter_prompt=brand_filter_prompt,
    filter_center=True,
    filter_iou=0.0,
)
annotated_brands = sam3_infer.annotate_brands(
    sample, brand_result["brands"]
)

print(f"Total brand detections: {brand_result['total_count']}")
for label, brand in brand_result["brands"].items():
    print(f"  {label}: {brand['count']} ({brand['prompt']})")
    if "raw_count" in brand:
        print(f"    model: {brand['raw_count']}  after cleanup: {brand['deduplicated_count']}  "
              f"redundant removed: {brand['redundant_box_count']}")
    if "filtered_count" in brand:
        print(f"    removed by filter: {brand['filtered_count']}")

display(annotated_brands)

## 8. Try your own images and prompts

Swap in your own file path, folder, or URL plus a verbose prompt and an optional filter prompt, then re-run the two cells below.

In [ ]:
my_image = "/kaggle/input/my-dataset/my_photo.jpg"   # edit me
my_prompt = "pairs of sunglasses displayed on the retail rack"  # edit me
my_filter_prompt = "faces of people"  # edit me (or None to disable filtering)

my_path = Path(my_image)
if not my_path.exists():
    print(f"File not found: {my_path}. Point `my_image` at a real file or folder.")
else:
    result = counter.infer(
        my_path, my_prompt,
        filter_prompt=my_filter_prompt,
        filter_center=True,
        filter_iou=0.0,
    )
    annotated = annotate(my_path, my_prompt, result["boxes"], result["scores"])
    print(f"Prompt: {my_prompt}")
    print(f"Filter: {my_filter_prompt!r}")
    print(f"Count:  {result['count']}")
    if "raw_count" in result:
        print(f"  model: {result['raw_count']}  after cleanup: {result['deduplicated_count']}  "
              f"enclosing/duplicate removed: {result['redundant_box_count']}")
    if "filtered_count" in result:
        print(f"  removed by filter: {result['filtered_count']}  filter objects: "
              f"{result['filter_object_count']}")
    display(annotated)

## 9. Cleanup

Releases the model from GPU memory when you are done.

In [ ]:
del counter, annotated, annotated_filtered, annotated_brands, brand_result
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("GPU cache cleared.")